In [ ]:
#!/usr/bin/env python3
import os
import csv
import wave
import whisper
import librosa
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from tqdm import tqdm

# Configuration
MODEL_SIZE = "small.en"          # Whisper model size
CHUNK_DURATION = 30               # Seconds per chunk
TARGET_SR = 22050                # Target sample rate (22050 Hz)
AUDIO_DIR = Path("./raw_cumberbatch_data")       # Input audio directory
CHUNKS_DIR = Path("./dataset/chunks")     # Output directory for chunks
CSV_PATH = Path("./dataset/transcriptions.csv")  # Output CSV file
STATE_FILE = Path("./dataset/processing_state.txt")  # Resume state tracking

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)

def load_resume_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, 'r') as f:
            return int(f.read().strip())
    return 0

def save_resume_state(last_index):
    with open(STATE_FILE, 'w') as f:
        f.write(str(last_index))

def convert_to_target_format(input_path, output_path):
    """Convert audio file to 22050Hz mono 16-bit PCM WAV"""
    try:
        audio = AudioSegment.from_file(input_path)
        audio = audio.set_frame_rate(TARGET_SR).set_channels(1)
        audio.export(output_path, format="wav", parameters=["-acodec", "pcm_s16le"])
        return True
    except Exception as e:
        print(f"Error converting {input_path}: {str(e)}")
        return False

def split_audio(file_path, chunk_index):
    """Split audio into 30-second chunks, return chunk paths and duration"""
    try:
        y, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
        duration = len(y) / sr
        chunk_size = CHUNK_DURATION * sr
        chunks = []
        
        for i in range(0, len(y), chunk_size):
            chunk = y[i:i+chunk_size]
            chunk_path = CHUNKS_DIR / f"{chunk_index:03d}.wav"
            
            with wave.open(str(chunk_path), 'wb') as wf:
                wf.setnchannels(1)
                wf.setsampwidth(2)  # 16-bit = 2 bytes
                wf.setframerate(TARGET_SR)
                wf.writeframes((chunk * 32767).astype(np.int16))
            
            chunks.append(chunk_path)
            chunk_index += 1
        
        return chunks, duration
    except Exception as e:
        print(f"Error splitting {file_path}: {str(e)}")
        return [], 0

def transcribe_chunk(model, audio_path):
    """Transcribe a single audio chunk with no progress bars"""
    # Whisper automatically resamples to 16kHz internally
    audio = whisper.load_audio(str(audio_path))
    result = model.transcribe(audio, fp16=False, verbose=None)
    return result["text"].strip()

def main():
    # Create directories if needed
    ensure_dir(AUDIO_DIR)
    ensure_dir(CHUNKS_DIR)
    
    # Initialize Whisper model
    model = whisper.load_model(MODEL_SIZE)
    
    # Resume state management
    chunk_index = load_resume_state()
    processed_files = set()
    
    # Load existing transcriptions if resuming
    if CSV_PATH.exists():
        with open(CSV_PATH, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)  # Skip header
            for row in reader:
                if row: processed_files.add(row[0])
    
    # Process audio files in directory
    audio_files = sorted(AUDIO_DIR.glob("*.wav"))
    
    # Create progress bar for files
    file_pbar = tqdm(audio_files, desc="Processing files")
    
    # Process each audio file
    with open(CSV_PATH, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        
        # Write header if new file
        if csvfile.tell() == 0:
            writer.writerow(["file_name", "transcription"])
        
        for audio_file in file_pbar:
            file_pbar.set_postfix(file=audio_file.name, current="Converting")
            
            # Convert to proper format if needed
            converted_path = CHUNKS_DIR / "temp.wav"
            if not convert_to_target_format(audio_file, converted_path):
                continue
            
            # Split into chunks and get audio duration
            chunks, duration = split_audio(converted_path, chunk_index)
            os.remove(converted_path)  # Cleanup temp file
            
            # Skip if no chunks generated
            if not chunks:
                continue
                
            # Calculate total chunks in this file
            total_chunks = len(chunks)
            file_pbar.set_postfix(
                file=audio_file.name,
                current="Transcribing",
                chunks=f"0/{total_chunks}",
                duration=f"{duration:.1f}s"
            )
            
            # Process each chunk with file-level progress bar
            for i, chunk_path in enumerate(chunks):
                chunk_name = chunk_path.name
                
                # Update file progress
                file_pbar.set_postfix(
                    file=audio_file.name,
                    current="Transcribing",
                    chunks=f"{i+1}/{total_chunks}",
                    duration=f"{duration:.1f}s"
                )
                
                # Skip already processed chunks
                if chunk_name in processed_files:
                    chunk_index += 1
                    continue
                
                # Transcribe and save
                try:
                    transcription = transcribe_chunk(model, chunk_path)
                    writer.writerow([chunk_name, transcription])
                    csvfile.flush()  # Ensure immediate write
                    processed_files.add(chunk_name)
                except Exception as e:
                    print(f"\nError transcribing {chunk_name}: {str(e)}")
                    save_resume_state(chunk_index)
                    return
                
                # Update state
                chunk_index += 1
                save_resume_state(chunk_index)
    
    # Cleanup state file after successful completion
    if STATE_FILE.exists():
        os.remove(STATE_FILE)
    
    print(f"\nProcessing complete! Results saved to {CSV_PATH}")

if __name__ == "__main__":
    main()

Processing files:   0%|          | 0/5 [03:01<?, ?it/s, chunks=62/606, current=Transcribing, duration=18167.1s, file=Audiobook - Benedict Cumberbatch read Casanova [oNhyLKUjRec].wav]